## Import Required Libraries

The libraries below are used for data handling, numerical calculations, and visualisation.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

## Dataset Loading and Initial Audit

The original Online Retail dataset is loaded for the micro-level analysis.

A random seed of 42 is set so that the results are reproducible.

The initial audit checks:
- Dataset shape
- Data types
- Missing values
- Summary statistics

In [ ]:
SEED = 42
np.random.seed(SEED)

# 1. Load the raw dataset
df = pd.read_csv(
    r'C:\Users\muthu\OneDrive\Desktop\ML-Capstone-Project\data\raw\data.csv',
    encoding='ISO-8859-1'
)

print("=" * 60)
print("SECTION A - DATASET AUDIT")
print("=" * 60)
print(f"Original Shape: {df.shape}")

SECTION A - DATASET AUDIT
Original Shape: (541909, 8)

Data Types & Missing Values:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB
None

Missing Value Counts:
 InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Summary Statistics:
                count          mean          std       min       25%      

### Dataset Structure and Missing Values

The dataset structure is examined to identify the data types of each column and the number of missing values.

This helps us understand the data before performing any cleaning or preprocessing.

In [ ]:
print("\nData Types and Dataset Information:")
df.info()

print("\nMissing Value Counts:")
print(df.isnull().sum())

print("\nSummary Statistics:")
print(df.describe().T)

## Initial Data Cleaning

Invalid transactions are removed before further analysis.

Rows with:
- Zero or negative quantity
- Zero or negative unit price

are excluded.

This removes cancelled/returned transactions and invalid pricing records from the modelling dataset.

In [ ]:
# Remove cancellations, returns, and invalid pricing records
df_clean = df[
    (df['Quantity'] > 0) &
    (df['UnitPrice'] > 0)
].copy()

print("Original rows:", len(df))
print("Rows after cleaning:", len(df_clean))
print("Rows removed:", len(df) - len(df_clean))

## Date Formatting

The `InvoiceDate` column is converted from text format to datetime format.

This allows us to extract useful date and time information later during feature engineering.

In [ ]:
df_clean['InvoiceDate'] = pd.to_datetime(
    df_clean['InvoiceDate'],
    errors='coerce'
)

print("InvoiceDate converted to datetime format.")
print(df_clean['InvoiceDate'].head())

## EDA Visualizations

The following visualizations are used to understand the distribution of the target variable, relationships among numerical variables, and differences in Unit Price across countries.

Since the transaction-level dataset contains a large number of observations, a sample is used for the scatter plot to make the visualization easier to interpret.

### Target Distribution

The histogram shows the distribution of `UnitPrice`. A logarithmic scale is used because Unit Price is highly skewed, with many low-priced transactions and relatively fewer high-priced transactions.

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(
    df_clean['UnitPrice'],
    kde=True,
    color='teal',
    log_scale=True
)

plt.title("Distribution of Unit Price (Log Scale)")
plt.xlabel("Unit Price ($)")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

### Correlation Heatmap

The correlation heatmap shows the linear relationships among the numerical variables in the cleaned dataset. It helps identify variables that have stronger relationships with the target variable.

In [ ]:
numeric_eda_cols = df_clean.select_dtypes(
    include=[np.number]
).columns

plt.figure(figsize=(8, 6))

sns.heatmap(
    df_clean[numeric_eda_cols].corr(),
    annot=True,
    cmap='mako',
    fmt=".2f"
)

plt.title("Feature Correlation Heatmap")

plt.tight_layout()
plt.show()

### Quantity vs. Unit Price

The scatter plot examines the relationship between `Quantity` and `UnitPrice`. Since the dataset contains a large number of transactions, a sample of up to 10,000 observations is used to improve readability.

In [ ]:
eda_sample = df_clean.sample(
    min(10000, len(df_clean)),
    random_state=SEED
)

plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=eda_sample,
    x='Quantity',
    y='UnitPrice',
    color='indigo',
    alpha=0.4
)

plt.title("Quantity vs. Unit Price (Sampled)")
plt.xlabel("Quantity")
plt.ylabel("Unit Price ($)")
plt.yscale('log')

plt.tight_layout()
plt.show()

### Unit Price Distribution Across Countries

The boxplot compares the distribution of `UnitPrice` across the six countries with the highest number of transactions. This helps identify differences in pricing patterns between major markets.

In [ ]:
top_countries = (
    df_clean['Country']
    .value_counts()
    .head(6)
    .index
)

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_clean[df_clean['Country'].isin(top_countries)],
    x='Country',
    y='UnitPrice',
    hue='Country',
    legend=False,
    palette='Set2'
)

plt.title("Unit Price Distribution Across Top Countries")
plt.xlabel("Country")
plt.ylabel("Unit Price ($)")
plt.xticks(rotation=45)
plt.yscale('log')

plt.tight_layout()
plt.show()

## Section B. Data Preprocessing

This section prepares the cleaned transaction-level data for machine learning.

The preprocessing steps include missing-value handling, duplicate removal, validity checks, date-based feature engineering, feature-target separation, stratified train-test splitting, categorical encoding, and numerical scaling.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

SEED = 42
TARGET = 'UnitPrice'

### B1. Dataset Overview

The cleaned transaction-level dataset from Section A is used for preprocessing. The dataset shape and available columns are checked before further processing.

In [ ]:
print("=" * 60)
print("INITIAL DATASET")
print("=" * 60)

print("Dataset shape:", df_clean.shape)

print("\nColumns:")
print(df_clean.columns.tolist())

### Missing Value Analysis

Missing values are examined before model training. Missing product descriptions are replaced with `UNKNOWN_PRODUCT`. `CustomerID` is not required for the model and will be removed later.

In [ ]:
print("=" * 60)
print("MISSING VALUE ANALYSIS")
print("=" * 60)

print(df_clean.isnull().sum())

### Handling Missing Product Descriptions

Missing values in `Description` are replaced with `UNKNOWN_PRODUCT`. This preserves the transaction records while providing a meaningful category for missing descriptions.

The `CustomerID` column will be removed later because it is an identifier rather than a useful predictive feature for this model.

In [ ]:
if 'Description' in df_clean.columns:
    df_clean['Description'] = df_clean['Description'].fillna(
        'UNKNOWN_PRODUCT'
    )

print("Missing Description values after handling:",
      df_clean['Description'].isnull().sum())

### Duplicate Handling

Duplicate rows are identified and removed to prevent repeated transaction records from influencing model training.

In [ ]:
duplicate_count = df_clean.duplicated().sum()

print("Duplicate rows found:", duplicate_count)

### Invalid Value and Physical Anomaly Handling

Transactions with non-positive `Quantity` or `UnitPrice` are removed because the prediction task considers valid sales transactions with positive quantity and price.

In [ ]:
initial_rows = len(df_clean)

df_clean = df_clean[
    (df_clean['Quantity'] > 0) &
    (df_clean['UnitPrice'] > 0)
].reset_index(drop=True)

removed_rows = initial_rows - len(df_clean)

print("Invalid records removed:", removed_rows)
print("Shape after validity filtering:", df_clean.shape)

### B1. Date Validation

`InvoiceDate` is converted into datetime format so that temporal information such as year, month, day of week, and hour can be extracted reliably.

In [ ]:
df_clean['InvoiceDate'] = pd.to_datetime(
    df_clean['InvoiceDate'],
    errors='coerce'
)

invalid_dates = df_clean['InvoiceDate'].isna().sum()

print("Invalid InvoiceDate values:", invalid_dates)

In [ ]:
df_clean = df_clean.dropna(
    subset=['InvoiceDate']
).reset_index(drop=True)

print("Shape after date cleaning:", df_clean.shape)

### Outlier Analysis

The Interquartile Range (IQR) method is used to identify potential outliers in `UnitPrice`.

High-priced transactions are not automatically removed because they may represent legitimate products. Instead, the distribution of the target will be handled using a logarithmic transformation.

In [ ]:
Q1 = df_clean[TARGET].quantile(0.25)
Q3 = df_clean[TARGET].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_mask = (
    (df_clean[TARGET] < lower_bound) |
    (df_clean[TARGET] > upper_bound)
)

outlier_count = outlier_mask.sum()

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of UnitPrice outliers:", outlier_count)

### Feature Engineering

Temporal features are extracted from `InvoiceDate` to capture potential purchasing patterns across different periods of the year and day.

The following features are created:
- `year`
- `month`
- `day_of_week`
- `hour`

In [ ]:
df_clean['year'] = df_clean['InvoiceDate'].dt.year
df_clean['month'] = df_clean['InvoiceDate'].dt.month
df_clean['day_of_week'] = df_clean['InvoiceDate'].dt.dayofweek
df_clean['hour'] = df_clean['InvoiceDate'].dt.hour

print("Created features:")
print("- year")
print("- month")
print("- day_of_week")
print("- hour")

### Removing Identifier and High-Cardinality Columns

Columns such as `InvoiceNo`, `StockCode`, `CustomerID`, and the original `InvoiceDate` are removed because they are identifiers or have high cardinality.

`Description` is also removed to avoid creating a very large categorical feature space.

The extracted temporal features are retained for modeling.

In [ ]:
df_clean = df_clean.drop(
    columns=[
        'InvoiceDate',
        'InvoiceNo',
        'StockCode',
        'Description',
        'CustomerID'
    ],
    errors='ignore'
)

print("Remaining columns:")
print(df_clean.columns.tolist())

### Separate Features and Target

`UnitPrice` is used as the regression target.

The original target values are preserved for final evaluation, while a logarithmic transformation is applied to the target to reduce the effect of its right-skewed distribution.

In [ ]:
X = df_clean.drop(columns=[TARGET])

y_original = df_clean[TARGET].copy()

y = np.log1p(y_original)

print("Number of features:", X.shape[1])
print("Target:", TARGET)

### Stratified Train-Test Split

The data is divided into 80% training data and 20% testing data.

Since `UnitPrice` is a continuous regression target, quantile bins are created to approximately preserve the target distribution in both sets.

The random seed is fixed at 42 for reproducibility.

In [ ]:
y_bins = pd.qcut(
    y,
    q=10,
    labels=False,
    duplicates='drop'
)

In [ ]:
(
    X_train,
    X_test,
    y_train,
    y_test,
    y_train_original,
    y_test_original
) = train_test_split(
    X,
    y,
    y_original,
    test_size=0.20,
    random_state=SEED,
    stratify=y_bins
)

print("=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

### Rare Category Handling

Rare countries are grouped into an `Other` category.

The country frequencies are calculated using only the training data to avoid information leakage from the test set.

In [ ]:
if 'Country' in X_train.columns:

    country_counts = X_train['Country'].value_counts()

    rare_countries = country_counts[
        country_counts < 5
    ].index

    X_train['Country'] = X_train['Country'].replace(
        rare_countries,
        'Other'
    )

    X_test['Country'] = X_test['Country'].replace(
        rare_countries,
        'Other'
    )

    print("Rare countries grouped:", len(rare_countries))

### Identify Numerical and Categorical Features

The features are separated into numerical and categorical groups.

Numerical features will be standardized using `StandardScaler`, while categorical features will be converted into numerical form using one-hot encoding.

In [ ]:
categorical_cols = X_train.select_dtypes(
    include=['object', 'category']
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    include=[np.number]
).columns.tolist()

print("Numerical features:")
print(numerical_cols)

print("\nCategorical features:")
print(categorical_cols)

### Scaling and Encoding

A `ColumnTransformer` is used to apply different preprocessing methods to different feature types.

- Numerical features are standardized using `StandardScaler`.
- Categorical features are one-hot encoded.
- The first category is dropped to avoid redundant dummy variables.
- Unknown categories in the test set are handled safely.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            numerical_cols
        ),
        (
            'cat',
            OneHotEncoder(
                drop='first',
                handle_unknown='ignore',
                sparse_output=False
            ),
            categorical_cols
        )
    ]
)

### Fit Preprocessing on Training Data Only

The scaler and encoder are fitted only on the training data.

The test data is transformed using the already-fitted preprocessing objects. This prevents information from the test set from influencing the training process.

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

### Final Processed Feature Matrices

The transformed numerical arrays are converted back into DataFrames with the generated feature names.

These processed matrices will be used as inputs for the regression models in Section C.

In [ ]:
feature_names = preprocessor.get_feature_names_out()

X_train_scaled = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print("Processed training data shape:", X_train_scaled.shape)
print("Processed testing data shape:", X_test_scaled.shape)

### Final Preprocessing Validation

The final preprocessing stage is validated by checking the processed dataset dimensions and confirming that no missing values remain in the training or testing feature matrices.

In [ ]:
print("=" * 60)
print("FINAL PREPROCESSING VALIDATION")
print("=" * 60)

print("Processed X_train shape:", X_train_scaled.shape)
print("Processed X_test shape :", X_test_scaled.shape)

print("\nMissing values in X_train:")
print(X_train_scaled.isnull().sum().sum())

print("\nMissing values in X_test:")
print(X_test_scaled.isnull().sum().sum())

### Target Transformation Validation

The original and log-transformed target distributions are compared using summary statistics to verify the effect of the logarithmic transformation.

In [ ]:
print("\nTarget transformation:")
print("Original target mean:", y_original.mean())
print("Original target median:", y_original.median())
print("Log-transformed target mean:", y.mean())

INITIAL DATASET
Dataset shape: (541909, 8)

Columns:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

MISSING VALUE ANALYSIS
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Duplicate rows found: 5268
Shape after duplicate removal: (536641, 8)

Invalid records removed: 11763
Shape after validity filtering: (524878, 8)

Invalid InvoiceDate values: 0
Shape after date cleaning: (524878, 8)

OUTLIER ANALYSIS
Q1: 1.25
Q3: 4.13
IQR: 2.88
Lower bound: -3.0700000000000003
Upper bound: 8.45
Number of UnitPrice outliers: 37827

FEATURE ENGINEERING
Created features:
- year
- month
- day_of_week
- hour

TRAIN / TEST SPLIT
X_train shape: (419902, 6)
X_test shape : (104976, 6)
y_train shape: (419902,)
y_test shape : (104976,)

Rare countries grouped: 0

Numerical features:
['Quantity', 'year', 'month', 

In [ ]:
print("\nSection B preprocessing completed successfully.")

print("Processed Features Head:")
print(X_train_scaled.head())